In [ ]:
import re
import pandas as pd
from sqlalchemy import create_engine
import requests
import time

engine = create_engine('sqlite:///terraria.db')
df = pd.read_excel('terraria-weapons.xlsx')

df.head()

,Image,Name,Damage,Damage type,Knockback,Critical chance,Use time,HMonly,Value,Stage
0,NaN,"Abigail's Flower (Desktop, Console and Mobile ...",6,Summon,2,-,36,❌,50 Silver,pre-boss
1,NaN,"Acorn Slingshot (Desktop, Console and Mobile v...",5,-,4,-,20,❌,20 Silver,pre-boss
2,NaN,Adamantite Chainsaw,33,Melee,4.5,4%,15,✔️,2 GC16 SC,pre-mech
3,NaN,Adamantite Drill,20,Melee,"0.5 (Extremely weak) (Desktop, Console and Mob...",4%,15,✔️,2 GC16 SC,pre-mech
4,NaN,Adamantite Glaive,49,Melee,6,4%,25,✔️,1 Gold 80 Silver,pre-mech


In [186]:
remove_items = [
    'Axe', 'Bomb', 'Hammer', 'Dynamite', 'Mine', 'Drill', 'Drax', 'Chainsaw', 'Reaver Shark', 'Digging Claw', 'Picksaw', 'Shovel', 'Blood Lust Cluster', 'Sawtooth Shark', 'Breaker', 'Flesh Grinder', 'Rockfish', 'Hammush', 'Regrowth', 'Red Rocket', 'Green Rocket', 'Blue Rocket', 'Yellow Rocket', 'Snowball Launcher'
]

df = df[
    ~df['Name'].str.contains(
        '|'.join(remove_items),
        case=False,
        na=False,
        regex=True
    )
]


exceptions = [
    'Charged Blaster Cannon',
    'KO Cannon',
    'Snowball Cannon',
    'Snowman Cannon',
    'Star Cannon'
]

df = df[
    ~df['Name'].str.contains('Cannon', case=False, na=False)
    | df['Name'].isin(exceptions)
]

In [187]:
df['Name'] = df['Name'].str.replace(" (Desktop, Console and Mobile versions)", "")
df['Name'] = df['Name'].str.replace(" (Desktop, Console, Old-gen console and Mobile versions)", "")

df['Damage'] = df['Damage'].str.replace(" (to enemies)1 (to user)", "")
df['Damage'] = pd.to_numeric(df['Damage'], errors="coerce").astype("Int64")

df['Knockback'] = df['Knockback'].str.replace(" (Extremely weak) (Desktop, Console and Mobile versions) / 0 (No knockback) (Old-gen console and 3DS versions)", "")
df['Knockback'] = pd.to_numeric(df['Knockback'], errors="coerce")

df['Critical chance'] = df['Critical chance'].str.replace("%", "")
df['Critical chance'] = pd.to_numeric(df['Critical chance'], errors="coerce").astype("Int64")

df['Use time'] = df['Use time'].str.replace(" (Insanely fast)(see notes for more info)", "")
df['Use time'] = df['Use time'].str.replace(" (Very fast) (Desktop, Console and Mobile versions) /8 (Insanely fast) (Old-gen console and 3DS versions)", "")
df['Use time'] = df['Use time'].str.replace(" (Very fast)(see notes for more info)", "")
df['Use time'] = pd.to_numeric(df['Use time'], errors="coerce").astype("Int64")

df['HMonly'] = df['HMonly'].map({
    "✔️" : True,
    "❌" : False
})


df['Value'] = df['Value'].str.replace(" (Desktop, Console and Mobile versions) / 20 Silver (Old-gen console and 3DS versions)", "")

df['Stage'] = df['Stage'].map({
    "pre-boss" : "1 pre-boss",
    "pre-hm" : "2 pre-hm",
    "pre-mech" : "3 pre-mech",
    "pre-plant" : "4 pre-plant",
    "pre-golem" : "5 pre-golem",
    "pre-lunar" : "6 pre-lunar",
    "post-moonlord" : "7 post-moonlord",
})

df.head()

AttributeError: Can only use .str accessor with string values, not integer

In [188]:
def convert_to_gold(value):
    currency_values ={
        'platinum': 100,
        'pc': 100,
        'gold': 1,
        'gc': 1,
        'silver': 0.01,
        'sc': 0.01,
        'copper': 0.0001,
        'cc': 0.0001
    }

    pattern = r"(\d+)\s*(platinum|gold|silver|copper|pc|gc|sc|cc)\b"

    total = 0.0

    for amount, currency in re.findall(pattern,str(value),re.IGNORECASE):
        total += int(amount) * currency_values[currency.lower()]

    return round(total, 4) if total > 0 else pd.NA

df['Value'] = df['Value'].apply(convert_to_gold)

df.head()

,Image,Name,Damage,Damage type,Knockback,Critical chance,Use time,HMonly,Value,Stage
0,https://terraria.wiki.gg/images/Abigail's_Flow...,Abigail's Flower,6,Summon,2.0,<NA>,36,False,<NA>,1 pre-boss
1,https://terraria.wiki.gg/images/Acorn_Slingsho...,Acorn Slingshot,5,-,4.0,<NA>,20,False,<NA>,1 pre-boss
4,https://terraria.wiki.gg/images/Adamantite_Gla...,Adamantite Glaive,49,Melee,6.0,4,25,True,<NA>,3 pre-mech
6,https://terraria.wiki.gg/images/Adamantite_Rep...,Adamantite Repeater,42,Ranged,2.5,4,18,True,<NA>,3 pre-mech
7,https://terraria.wiki.gg/images/Adamantite_Swo...,Adamantite Sword,61,Melee,6.0,4,21,True,<NA>,3 pre-mech


In [189]:
BASE_URL = 'https://terraria.wiki.gg/images/'

# Create the column if needed, preserving existing results
if 'Image' not in df.columns:
    df['Image'] = pd.Series(index=df.index, dtype='object')
else:
    df['Image'] = df['Image'].astype('object')

for index, row in df.iterrows():

    # Skip weapons already checked successfully
    if pd.notna(row['Image']):
        continue

    weapon = row['Name'].replace(" ", "_").replace(":","_").replace("Trusty_", "")
    

    url = BASE_URL + weapon + '.png'
    response = requests.get(url, timeout=15)
    print(url)
    print(response.status_code)

    # Respect rate limits and stop safely
    if response.status_code == 429:
        wait = response.headers.get('Retry-After', '60')
        print(f"Rate limited. Retry after {wait} seconds.")
        print(f"Stopped at row {index}. Rerun after the cooldown.")
        break

    if response.status_code == 200:
        df.at[index, 'Image'] = url

    elif response.status_code == 404:
        url = BASE_URL + weapon + '.gif'
        response = requests.get(url, timeout=15)

        if response.status_code == 429:
            wait = response.headers.get('Retry-After', '60')
            print(f"Rate limited. Retry after {wait} seconds.")
            print(f"Stopped at row {index}. Rerun after the cooldown.")
            break

        if response.status_code == 200:
            df.at[index, 'Image'] = url

    # Pause between weapons
    time.sleep(2)

print("Image lookup pass finished.")
print("Images found:", df['Image'].notna().sum())
print("Images remaining:", df['Image'].isna().sum())

Image lookup pass finished.
Images found: 454
Images remaining: 0


In [136]:
# from sqlalchemy import text

# with engine.begin() as conn:
#     conn.execute(text("""
#         CREATE VIEW IF NOT EXISTS items_sorted AS
#         SELECT *,
#           CASE Stage
#             WHEN 'pre-boss' THEN 1 WHEN 'pre-hm' THEN 2
#             WHEN 'pre-mech' THEN 3 WHEN 'pre-plant' THEN 4
#             WHEN 'pre-golem' THEN 5 WHEN 'pre-lunar' THEN 6
#             WHEN 'post-moonlord' THEN 7 ELSE 99
#           END AS stage_rank
#         FROM weapons
#         ORDER BY Name COLLATE NOCASE
#     """))

In [ ]:
df.to_sql(
    name='weapons',
    con=engine,
    if_exists='replace',
    index=False
)

454